# ARPG-L Baseline Reproduction (Colab)

**Before running:** Go to **Runtime > Change runtime type** and select **A100 GPU** (or T4/V100 if A100 is unavailable).

This notebook:
1. Verifies GPU access
2. Mounts Google Drive (assets + results persist across sessions)
3. Clones the repo & installs deps
4. Downloads model weights + reference batch (first time only)
5. Runs a 1k smoke test
6. Runs the full 50k-sample baseline
7. Computes FID (target: ~2.38)

## 1. Verify GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > A100 GPU")

## 2. Mount Google Drive

All heavy data (weights, reference batch, results) is stored on Drive so you don't re-download every session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ARPG = "/content/drive/MyDrive/ARPG-assets"
!mkdir -p {DRIVE_ARPG}/weights {DRIVE_ARPG}/eval {DRIVE_ARPG}/external {DRIVE_ARPG}/results

## 3. Clone repo & install dependencies

In [ ]:
# Clone repo (edit URL if needed)
!rm -rf /content/ARPG
!git clone https://github.com/rshahbazov23/comp447-arpg-private.git /content/ARPG
%cd /content/ARPG

# Symlink heavy directories to Google Drive
!ln -sfn {DRIVE_ARPG}/weights weights
!ln -sfn {DRIVE_ARPG}/eval eval
!ln -sfn {DRIVE_ARPG}/external external
!ln -sfn {DRIVE_ARPG}/results results

# Install runtime deps (PyTorch is already in Colab)
!pip install -q transformers einops Pillow tqdm numpy

In [ ]:
# Verify the repo imports work
import sys, subprocess
result = subprocess.run(
    [sys.executable, "-c",
     "from models.arpg import ARPG_models; print('Available models:', sorted(ARPG_models.keys()))"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)
    raise RuntimeError("Model import failed")

## 4. Download assets (first time only)

Downloads ~2.5 GB to Google Drive. If the files already exist on Drive, this is skipped automatically.

In [ ]:
# Downloads to Drive via symlinks. Already-downloaded files are skipped.
!bash scripts/download_baseline_assets.sh

In [ ]:
# Verify all assets are present
import os
assets = [
    "weights/arpg_300m.pt",
    "weights/vq_ds16_c2i.pt",
    "eval/VIRTUAL_imagenet256_labeled.npz",
    "external/guided-diffusion/evaluations/evaluator.py",
]
for path in assets:
    exists = os.path.exists(path)
    size = f"{os.path.getsize(path)/1e6:.1f} MB" if exists else "MISSING"
    status = "OK" if exists else "FAIL"
    print(f"  [{status}] {path}  ({size})")

## 5. Smoke test (1k samples)

Quick sanity check. ~2-3 min on A100, ~5-10 min on T4.
Uses `--no-compile` to avoid potential `torch.compile` issues on Colab.
Samples are saved to local disk (fast I/O), then backed up to Drive after.

In [ ]:
!ARPG_NO_COMPILE=1 SAMPLE_DIR=/content/samples/arpgl-baseline bash scripts/run_arpgl_smoke.sh

In [ ]:
# Check that samples were generated
import glob
smoke_dir = "/content/samples/arpgl-baseline"
sample_dirs = glob.glob(f"{smoke_dir}/ARPG-L-*")
if sample_dirs:
    pngs = glob.glob(f"{sample_dirs[0]}/*.png")
    print(f"Generated {len(pngs)} sample images in {sample_dirs[0]}")
    npzs = glob.glob(f"{smoke_dir}/*.npz")
    if npzs:
        print(f"NPZ file: {npzs[0]}")
else:
    print("ERROR: No sample directory found!")

In [ ]:
# Preview a few generated samples
from PIL import Image
import matplotlib.pyplot as plt

if sample_dirs:
    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    for i, ax in enumerate(axes):
        img = Image.open(f"{sample_dirs[0]}/{i:06d}.png")
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f"Sample {i}")
    plt.suptitle("Smoke test samples")
    plt.tight_layout()
    plt.show()

## 6. Full 50k-sample baseline

This is the real run. Time estimates (1 GPU):
- **A100**: ~30-45 min
- **V100**: ~1-2 hours
- **T4**: ~2-4 hours

Samples go to local `/content/samples/` for fast I/O, then the NPZ is copied to Drive.

In [ ]:
# Clean up smoke test output
!rm -rf /content/samples/arpgl-baseline

In [ ]:
!ARPG_NO_COMPILE=1 SAMPLE_DIR=/content/samples/arpgl-baseline bash scripts/run_arpgl_full.sh

In [ ]:
# Verify 50k samples + NPZ, then back up NPZ to Drive
import glob, shutil, numpy as np

sample_dirs = glob.glob("/content/samples/arpgl-baseline/ARPG-L-*")
if sample_dirs:
    pngs = glob.glob(f"{sample_dirs[0]}/*.png")
    print(f"Generated {len(pngs)} sample images")

npzs = glob.glob("/content/samples/arpgl-baseline/*.npz")
if npzs:
    data = np.load(npzs[0])
    print(f"NPZ shape: {data['arr_0'].shape}")
    print(f"NPZ path: {npzs[0]}")
    # Back up to Google Drive
    dest = f"{DRIVE_ARPG}/results/{os.path.basename(npzs[0])}"
    shutil.copy2(npzs[0], dest)
    print(f"Backed up to: {dest}")

## 7. Compute FID

Uses OpenAI's `guided-diffusion` evaluator. Target FID: **~2.38** (acceptable: 2.2-2.6).

In [ ]:
# Install eval deps (scipy and tensorflow for the evaluator)
!pip install -q scipy tensorflow

In [ ]:
!SAMPLE_DIR=/content/samples/arpgl-baseline bash scripts/eval_arpgl_baseline.sh